# 面试问题：RLOO 怎样用同 Prompt 的其他采样构造无 Value Model 的策略梯度基线？

        ## 可直接复述的回答主线

        1. RLOO 为同一 prompt 采样 K 条回复，并用其余 K-1 条奖励均值作为当前回复的 baseline。
2. 与把自身也放进组均值相比，leave-one-out 不会让当前奖励同时污染自己的基线。
3. 每条回复的 advantage 等于奖励减去其他样本均值，同组 advantage 总和仍为零。
4. 策略目标应使用可读回复、序列 log probability、长度、奖励分项和 KL，而不是裸布尔值。
5. K 等于一时没有可用的 leave-one-out baseline，必须拒绝该组或回退到独立运行均值。
6. 生产训练还要治理奖励作弊、组内相关采样、长度偏差、跨卡 group 对齐和参考策略漂移。

        后续代码会在同一批输入上展示朴素基线、底层计算、逐步轨迹、失败复现和修正结果。

## 1. 真实案例与输入预览

案例包含五个中文客服与推理 prompt，每个 prompt 固定四条可读候选回复。奖励由正确性、合规性和格式规则离线合成，结构与真实 RLOO rollout 一致但不代表基础模型泛化；所有文本均为脱敏教学样本。

In [1]:
import torch  # 使用基础张量运算手写 RLOO advantage 和策略目标。
groups = [{"id": "rloo-01", "prompt": "退款多久到账？", "responses": ["三个工作日到账", "立即到账", "无法退款", "请提供密码"], "rewards": [1.0, 0.2, -0.3, -1.0]}, {"id": "rloo-02", "prompt": "24箱每箱18件，共多少件？", "responses": ["432件", "422件", "42件", "无法计算"], "rewards": [1.0, 0.0, -0.2, -0.3]}, {"id": "rloo-03", "prompt": "用户索要他人手机号，应如何处理？", "responses": ["拒绝并说明权限", "直接导出", "只给后四位", "转交人工审核"], "rewards": [1.0, -1.0, -0.6, 0.7]}, {"id": "rloo-04", "prompt": "如何关闭会员自动续费？", "responses": ["在设置页关闭", "删除应用即可", "无需处理", "联系客服关闭"], "rewards": [0.9, -0.2, -0.4, 0.6]}, {"id": "rloo-05", "prompt": "只输出JSON状态ok。", "responses": ['{"status":"ok"}', "status=ok", '{"ok":true}', "好的"], "rewards": [1.0, 0.2, 0.1, -0.2]}]  # 定义五组有业务语义的 prompt 和四条候选奖励。
group_size = 4  # 固定每个 prompt 的 rollout 数以便 leave-one-out。
rewards = torch.tensor([reward for group in groups for reward in group["rewards"]], dtype=torch.float64)  # 展平二十条序列奖励。
sequence_logp = torch.linspace(-2.2, -0.7, steps=len(rewards), dtype=torch.float64, requires_grad=True)  # 构造当前策略对二十条回复的确定性序列 log probability。
reference_logp = sequence_logp.detach() - torch.tensor([0.08, 0.03, 0.05, 0.02] * len(groups), dtype=torch.float64)  # 构造冻结参考策略用于 KL 代理。
lengths = torch.tensor([6, 4, 4, 5, 3, 3, 3, 4, 6, 4, 5, 5, 5, 5, 4, 4, 7, 4, 5, 2], dtype=torch.float64)  # 记录每条可读回复的有效 Token 长度代理。
print("教学实验输入：五个 Prompt，每组四条回复")  # 标记下方为离线 RLOO rollout。
for group in groups:  # 逐组展示 prompt、回复和奖励。
    print(f"{group['id']} | {group['prompt']}")  # 输出当前 prompt。
    print("  ", list(zip(group["responses"], group["rewards"])))  # 输出当前组可读回复与奖励。
print("reward shape=", tuple(rewards.shape), "group_size=", group_size)  # 展示训练张量与 group 结构。

教学实验输入：五个 Prompt，每组四条回复
rloo-01 | 退款多久到账？
   [('三个工作日到账', 1.0), ('立即到账', 0.2), ('无法退款', -0.3), ('请提供密码', -1.0)]
rloo-02 | 24箱每箱18件，共多少件？
   [('432件', 1.0), ('422件', 0.0), ('42件', -0.2), ('无法计算', -0.3)]
rloo-03 | 用户索要他人手机号，应如何处理？
   [('拒绝并说明权限', 1.0), ('直接导出', -1.0), ('只给后四位', -0.6), ('转交人工审核', 0.7)]
rloo-04 | 如何关闭会员自动续费？
   [('在设置页关闭', 0.9), ('删除应用即可', -0.2), ('无需处理', -0.4), ('联系客服关闭', 0.6)]
rloo-05 | 只输出JSON状态ok。
   [('{"status":"ok"}', 1.0), ('status=ok', 0.2), ('{"ok":true}', 0.1), ('好的', -0.2)]
reward shape= (20,) group_size= 4


## 2. Baseline / 基线：把当前样本也放进组均值

朴素组均值实现很直观，但当前奖励会改变自己的 baseline，使 advantage 绝对值缩小。下面展示第一组四条回复的这种自污染。

In [2]:
grouped_rewards = rewards.reshape(len(groups), group_size)  # 恢复 prompt 乘采样数的二维奖励矩阵。
inclusive_baseline = grouped_rewards.mean(dim=1, keepdim=True)  # 计算包含当前样本自身的组均值。
inclusive_advantage = grouped_rewards - inclusive_baseline  # 构造朴素组中心化 advantage。
print("Baseline 第一组：包含自身的组均值")  # 标记当前输出属于自污染基线。
print("回复                 reward  baseline  advantage")  # 输出回复级表头。
for offset, response in enumerate(groups[0]["responses"]):  # 逐条展示第一组四个候选。
    print(f"{response:<20} {grouped_rewards[0, offset].item():>6.2f} {inclusive_baseline[0, 0].item():>9.3f} {inclusive_advantage[0, offset].item():>10.3f}")  # 输出当前回复的朴素优势。

Baseline 第一组：包含自身的组均值
回复                 reward  baseline  advantage
三个工作日到账                1.00    -0.025      1.025
立即到账                   0.20    -0.025      0.225
无法退款                  -0.30    -0.025     -0.275
请提供密码                 -1.00    -0.025     -0.975


## 3. 底层实现：Leave-One-Out baseline、长度归一化与 KL

对每个位置，用组奖励总和减去当前奖励，再除以 K-1。策略损失使用长度归一化 log probability，避免长回复仅因 Token 更多而占据更大尺度。

In [3]:
def rloo_advantages(group_reward_matrix):  # 手写每个样本排除自身的奖励基线。
    sample_count = group_reward_matrix.shape[1]  # 读取每组采样数量。
    if sample_count < 2:  # K 小于二时没有其他样本可作 baseline。
        raise ValueError("RLOO 每组至少需要两个采样")  # 显式拒绝无法定义的组。
    group_sum = group_reward_matrix.sum(dim=1, keepdim=True)  # 计算每个 prompt 的奖励总和。
    leave_one_out_baseline = (group_sum - group_reward_matrix) / (sample_count - 1)  # 对每条回复排除自身后求均值。
    advantage = group_reward_matrix - leave_one_out_baseline  # 计算当前奖励相对其他回复的优势。
    return advantage, leave_one_out_baseline  # 返回逐回复优势和可审计 baseline。
rloo_advantage_matrix, rloo_baseline_matrix = rloo_advantages(grouped_rewards)  # 对五组 rollout 计算 RLOO 信号。
flat_advantage = rloo_advantage_matrix.reshape(-1).detach()  # 展平并阻断奖励侧梯度。
normalized_logp = sequence_logp / lengths  # 按回复长度归一化序列 log probability。
sampled_kl = sequence_logp - reference_logp  # 计算当前策略相对参考策略的采样 KL 代理。
beta = 0.04  # 设置小型教学 KL 惩罚系数。
policy_term = -(flat_advantage * normalized_logp).mean()  # 计算 RLOO REINFORCE 策略项。
kl_term = beta * sampled_kl.mean()  # 计算参考策略 KL 惩罚项。
total_loss = policy_term + kl_term  # 合并策略梯度与 KL 得到训练目标。
print("RLOO 第一组逐回复中间量")  # 选择第一组解释排除自身后的变化。
print("回复                 reward  LOO基线  RLOO优势  长度  norm_logp")  # 输出底层计算表头。
for offset, response in enumerate(groups[0]["responses"]):  # 逐条展示第一组 RLOO 信号。
    flat_index = offset  # 第一组在展平张量中的偏移等于组内位置。
    print(f"{response:<20} {grouped_rewards[0, offset].item():>6.2f} {rloo_baseline_matrix[0, offset].item():>8.3f} {rloo_advantage_matrix[0, offset].item():>9.3f} {lengths[flat_index].item():>5.0f} {normalized_logp[flat_index].item():>10.3f}")  # 输出当前回复的 baseline、优势和策略尺度。

RLOO 第一组逐回复中间量
回复                 reward  LOO基线  RLOO优势  长度  norm_logp
三个工作日到账                1.00   -0.367     1.367     6     -0.367
立即到账                   0.20   -0.100     0.300     4     -0.530
无法退款                  -0.30    0.067    -0.367     4     -0.511
请提供密码                 -1.00    0.300    -1.300     5     -0.393


## 4. 结果表与结果解读

包含自身和 leave-one-out 的优势方向相同，但 RLOO 绝对值更大，因为当前奖励不再拉动自己的 baseline。五组 advantage 和都为零，因此不会整体抬高某个 prompt。

In [4]:
print("group       inclusive绝对和  RLOO绝对和  RLOO和  最优回复")  # 输出两种 advantage 的组级对照表头。
for group_index, group in enumerate(groups):  # 逐组比较信号尺度与最优回复。
    inclusive_magnitude = inclusive_advantage[group_index].abs().sum().item()  # 汇总包含自身基线的优势绝对值。
    rloo_magnitude = rloo_advantage_matrix[group_index].abs().sum().item()  # 汇总 leave-one-out 优势绝对值。
    best_index = int(torch.argmax(rloo_advantage_matrix[group_index]).item())  # 找到组内最大相对优势回复。
    print(f"{group['id']:<11} {inclusive_magnitude:>15.3f} {rloo_magnitude:>11.3f} {rloo_advantage_matrix[group_index].sum().item():>7.3f}  {group['responses'][best_index]}")  # 输出组级比较结果。
total_loss.backward()  # 对手写目标执行一次反向传播以观察真实梯度。
gradient = sequence_logp.grad.detach()  # 读取二十条回复 log probability 的梯度。
print(f"结果解读：policy={policy_term.item():.4f}，KL={kl_term.item():.4f}，loss={total_loss.item():.4f}；第一组最佳回复梯度={gradient[0].item():.4f}。")  # 解释策略方向和 KL 贡献。

group       inclusive绝对和  RLOO绝对和  RLOO和  最优回复
rloo-01               2.500       3.333   0.000  三个工作日到账
rloo-02               1.750       2.333   0.000  432件
rloo-03               3.300       4.400   0.000  拒绝并说明权限
rloo-04               2.100       2.800   0.000  在设置页关闭
rloo-05               1.450       1.933   0.000  {"status":"ok"}
结果解读：policy=-0.0105，KL=0.0018，loss=-0.0087；第一组最佳回复梯度=-0.0094。


## 5. 失败案例与修正

每个 prompt 只有一条采样时，K-1 为零，RLOO baseline 没有定义。错误做法是静默除零；修正是准入时要求 K≥2，或明确使用跨批次运行 baseline。

In [5]:
single_reward = torch.tensor([[0.8]], dtype=torch.float64)  # 构造只有一条回复的非法 RLOO 组。
naive_denominator = single_reward.shape[1] - 1  # 计算错误实现会使用的零分母。
failure_message = None  # 保存显式门禁返回的错误说明。
try:  # 调用核心实现验证 K 等于一会被拒绝。
    rloo_advantages(single_reward)  # 尝试对非法单样本组计算 RLOO。
except ValueError as error:  # 捕获预期的组大小门禁。
    failure_message = str(error)  # 保存可读错误而不是产生 NaN。
fallback_baseline = rewards.mean().item()  # 演示可选的独立运行均值回退值。
print(f"错误行为：K=1 时朴素分母={naive_denominator}，leave-one-out 不可定义。")  # 展示数学失败原因。
print(f"修正行为：门禁消息={failure_message}；若产品允许可显式回退运行baseline={fallback_baseline:.3f}。")  # 展示拒绝和可配置回退。

错误行为：K=1 时朴素分母=0，leave-one-out 不可定义。
修正行为：门禁消息=RLOO 每组至少需要两个采样；若产品允许可显式回退运行baseline=0.125。


## 6. 生产边界

实际 RLOO 需要真实模型采样和 Token 级 KL，还要保证同一 prompt 的 K 条轨迹在分布式 batch 中完整。应监控组内奖励方差、重复回复率、长度、KL、奖励分项和 K=1 丢弃率。

In [6]:
duplicate_rate = sum(len(group["responses"]) - len(set(group["responses"])) for group in groups) / (len(groups) * group_size)  # 统计教学 rollout 的完全重复回复比例。
reward_std = grouped_rewards.std(dim=1, unbiased=False)  # 计算每个 prompt 的组内奖励标准差。
diagnostics = {"groups": len(groups), "group_size": group_size, "mean_group_reward_std": reward_std.mean().item(), "duplicate_rate": duplicate_rate, "mean_sampled_kl": sampled_kl.mean().item()}  # 汇总生产训练应持续观察的指标。
print("生产监控快照：", {key: round(value, 4) if isinstance(value, float) else value for key, value in diagnostics.items()})  # 输出可读的组级训练诊断。

生产监控快照： {'groups': 5, 'group_size': 4, 'mean_group_reward_std': 0.6147, 'duplicate_rate': 0.0, 'mean_sampled_kl': 0.045}


## 7. 最小回归测试

只验证案例规模、LOO 公式、组和、梯度方向和 K=1 门禁。

In [7]:
assert len(groups) >= 5  # 保证案例至少包含五个有语义的 prompt。
assert torch.allclose(rloo_baseline_matrix[0, 0], grouped_rewards[0, 1:].mean())  # 保证第一条回复 baseline 真正排除了自身。
assert torch.allclose(rloo_advantage_matrix.sum(dim=1), torch.zeros(len(groups), dtype=torch.float64), atol=1.0e-12)  # 保证每组 RLOO 优势和为零。
assert gradient[0] < 0.0  # 保证梯度下降会提高第一组最佳回复的 log probability。
assert failure_message is not None  # 保证单样本组被显式门禁而非静默产生非数值。